# 03 · Gold — Indicadores analíticos Spark SQL

**Proyecto:** PoC Arquitectura Lakehouse para Monitoreo Ambiental Hídrico  
**Autora:** Kathiana Murillo García  
**Máster:** UNIR — Análisis y Visualización de Datos Masivos (2026–2027)  

## Objetivo
Construir dos indicadores analíticos Gold a partir de los registros VALID de la capa Silver:

- **Indicador 1 (PI2):** Índice de Oxigenación Mensual por Estación  
- **Indicador 2 (PI3):** Tendencia de Eutrofización Anual (T-P, Chl-a)

Ambos indicadores se construyen con **Spark SQL** sobre la vista temporal
`silver_kasumigaura`, creada a partir de los registros VALID de la tabla Silver.

_Revisado: umbrales OECD confirmados para lago templado._

## 1. Configuración

In [ ]:
# Celda 1 — Parámetros
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd

CATALOG = "training"
SCHEMA  = "kasumigaura"
TABLE_SILVER = f"{CATALOG}.{SCHEMA}.silver_water_quality"
TABLE_GOLD_OX = f"{CATALOG}.{SCHEMA}.gold_oxigenacion_mensual"
TABLE_GOLD_EU = f"{CATALOG}.{SCHEMA}.gold_eutrofizacion_anual"

try:
    CATALOG = spark.sql("SELECT current_catalog()").collect()[0][0]
    TABLE_SILVER = f"{CATALOG}.{SCHEMA}.silver_water_quality"
    TABLE_GOLD_OX = f"{CATALOG}.{SCHEMA}.gold_oxigenacion_mensual"
    TABLE_GOLD_EU = f"{CATALOG}.{SCHEMA}.gold_eutrofizacion_anual"
except Exception:
    pass

print(f"Silver : {TABLE_SILVER}")
print(f"Gold Ox: {TABLE_GOLD_OX}")
print(f"Gold Eu: {TABLE_GOLD_EU}")

## 2. Vista temporal de registros VALID

In [ ]:
# Celda 2 — Crear vista temporal `silver_kasumigaura` con solo registros VALID
spark.table(TABLE_SILVER) \
    .filter(F.col("dq_flag") == "VALID") \
    .createOrReplaceTempView("silver_kasumigaura")

n_valid = spark.sql("SELECT COUNT(*) FROM silver_kasumigaura").collect()[0][0]
print(f"Vista silver_kasumigaura: {n_valid:,} registros VALID")

## 3. Indicador 1 — Índice de Oxigenación Mensual por Estación

In [ ]:
# Celda 3 — Construir Indicador 1 con Spark SQL
# Clasifica el estado ecológico según umbrales limnológicos (Wetzel, 2001):
#   hipoxia_critica : DO0m < 4 mg/L
#   aceptable       : 4 ≤ DO0m < 6 mg/L
#   optimo          : DO0m ≥ 6 mg/L

spark.sql(f"""
CREATE OR REPLACE TABLE {TABLE_GOLD_OX} AS
SELECT
    `ST.`                               AS station,
    YEAR(Date)                          AS year,
    MONTH(Date)                         AS month,
    ROUND(AVG(DO0m), 2)                 AS avg_do_mg_l,
    ROUND(MIN(DO0m), 2)                 AS min_do_mg_l,
    ROUND(MAX(DO0m), 2)                 AS max_do_mg_l,
    ROUND(STDDEV(DO0m), 3)              AS std_do_mg_l,
    COUNT(*)                            AS n_obs,
    CASE
        WHEN AVG(DO0m) < 4  THEN 'hipoxia_critica'
        WHEN AVG(DO0m) < 6  THEN 'aceptable'
        ELSE 'optimo'
    END                                 AS estado_oxigenacion
FROM silver_kasumigaura
WHERE DO0m IS NOT NULL
GROUP BY `ST.`, YEAR(Date), MONTH(Date)
ORDER BY station, year, month
""")

n_ox = spark.table(TABLE_GOLD_OX).count()
print(f"✓ {TABLE_GOLD_OX}: {n_ox:,} registros")
spark.table(TABLE_GOLD_OX).show(5)

## 4. Indicador 2 — Tendencia de Eutrofización Anual

In [ ]:
# Celda 4 — Construir Indicador 2 con Spark SQL
# Solo registros desde 1984 (año en que T-P y Chl-a empezaron a medirse
# de forma sistemática en el programa NIES).

spark.sql(f"""
CREATE OR REPLACE TABLE {TABLE_GOLD_EU} AS
SELECT
    `ST.`                               AS station,
    YEAR(Date)                          AS year,
    ROUND(AVG(`T-P`),  3)               AS avg_tp_ug_l,
    ROUND(AVG(`T-N`),  1)               AS avg_tn_ug_l,
    ROUND(AVG(`Chl-a`), 2)              AS avg_chla_ug_l,
    COUNT(*)                            AS n_obs_valid,
    CASE
        WHEN AVG(`T-P`) < 10   THEN 'oligotrofico'
        WHEN AVG(`T-P`) < 35   THEN 'mesotrofico'
        WHEN AVG(`T-P`) < 100  THEN 'eutrofico'
        ELSE 'hipereutrofico'
    END                                 AS estado_trofico_tp
FROM silver_kasumigaura
WHERE YEAR(Date) >= 1984
  AND `T-P` IS NOT NULL
  AND `Chl-a` IS NOT NULL
GROUP BY `ST.`, YEAR(Date)
ORDER BY station, year
""")

n_eu = spark.table(TABLE_GOLD_EU).count()
print(f"✓ {TABLE_GOLD_EU}: {n_eu:,} registros")
spark.table(TABLE_GOLD_EU).show(5)

## 5. Visualizaciones con matplotlib

In [ ]:
# Celda 5 — Figura 1: Índice de Oxigenación Anual promedio
df_ox_pd = (
    spark.table(TABLE_GOLD_OX)
    .groupBy("year")
    .agg(F.round(F.avg("avg_do_mg_l"), 2).alias("do_anual"))
    .orderBy("year")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(df_ox_pd["year"], df_ox_pd["do_anual"],
        color="#2196F3", linewidth=2, marker="o", markersize=4, label="DO promedio anual")

ax.axhline(y=6, color="#4CAF50", linestyle="--", linewidth=1.5, label="Nivel aceptable (6 mg/L)")
ax.axhline(y=4, color="#F44336", linestyle="--", linewidth=1.5, label="Hipoxia crítica (4 mg/L)")
ax.fill_between(df_ox_pd["year"], 0, 4, alpha=0.07, color="#F44336", label="Zona hipoxia")

ax.set_title("Índice de Oxigenación Anual — Lago Kasumigaura (1977–2024)",
             fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Año", fontsize=12)
ax.set_ylabel("Oxígeno disuelto promedio (mg/L)", fontsize=12)
ax.legend(fontsize=10, loc="lower right")
ax.grid(True, alpha=0.3)
ax.set_xlim(df_ox_pd["year"].min() - 1, df_ox_pd["year"].max() + 1)
ax.xaxis.set_major_locator(ticker.MultipleLocator(10))

plt.tight_layout()
plt.savefig("/tmp/grafico_oxigenacion.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: /tmp/grafico_oxigenacion.png")

In [ ]:
# Celda 6 — Figura 2: Tendencia de Eutrofización Anual
df_eu_pd = (
    spark.table(TABLE_GOLD_EU)
    .groupBy("year")
    .agg(
        F.round(F.avg("avg_tp_ug_l"),  2).alias("tp_anual"),
        F.round(F.avg("avg_chla_ug_l"), 2).alias("chla_anual")
    )
    .orderBy("year")
    .toPandas()
)

fig, ax1 = plt.subplots(figsize=(12, 6))
ax2 = ax1.twinx()

ax1.plot(df_eu_pd["year"], df_eu_pd["tp_anual"],
         color="#E91E63", linewidth=2, marker="o", markersize=3,
         label="Fósforo total T-P (µg/L)")
ax2.plot(df_eu_pd["year"], df_eu_pd["chla_anual"],
         color="#4CAF50", linewidth=2, linestyle="--", marker="s", markersize=3,
         label="Clorofila-a (µg/L)")

ax1.set_title("Tendencia de Eutrofización Anual — Lago Kasumigaura (1984–2024)",
              fontsize=14, fontweight="bold", pad=12)
ax1.set_xlabel("Año", fontsize=12)
ax1.set_ylabel("Fósforo total T-P (µg/L)", fontsize=12, color="#E91E63")
ax2.set_ylabel("Clorofila-a (µg/L)", fontsize=12, color="#4CAF50")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10, loc="upper right")
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_locator(ticker.MultipleLocator(5))

plt.tight_layout()
plt.savefig("/tmp/grafico_eutrofizacion.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: /tmp/grafico_eutrofizacion.png")

## 6. Verificación end-to-end del pipeline

In [ ]:
# Celda 7 — Verificación de integridad referencial entre capas
CATALOG_DOT = f"{CATALOG}.{SCHEMA}"

n_bronze = spark.sql(f"SELECT COUNT(*) FROM {CATALOG_DOT}.bronze_water_quality").collect()[0][0]
n_silver = spark.sql(f"SELECT COUNT(*) FROM {CATALOG_DOT}.silver_water_quality").collect()[0][0]
n_silver_valid = spark.sql(
    f"SELECT COUNT(*) FROM {CATALOG_DOT}.silver_water_quality WHERE dq_flag = 'VALID'"
).collect()[0][0]
n_gold_ox = spark.sql(f"SELECT COUNT(*) FROM {TABLE_GOLD_OX}").collect()[0][0]
n_gold_eu = spark.sql(f"SELECT COUNT(*) FROM {TABLE_GOLD_EU}").collect()[0][0]

print("── Verificación end-to-end del pipeline ──────────────────────────")
print(f"Bronze                     : {n_bronze:>8,} registros")
print(f"Silver (total)             : {n_silver:>8,} registros")
print(f"Silver VALID               : {n_silver_valid:>8,} registros ({n_silver_valid/n_silver*100:.1f}%)")
print(f"Gold — Oxigenación mensual : {n_gold_ox:>8,} registros")
print(f"Gold — Eutrofización anual : {n_gold_eu:>8,} registros")
print()

assert n_bronze == n_silver, "ERROR: Bronze != Silver (pérdida de registros)"
print("✓ Bronze → Silver: conservación perfecta")
print(f"✓ Ratio compresión Ox : {n_silver_valid/n_gold_ox:.2f} registros VALID/agregado mensual")
print(f"✓ Ratio compresión Eu : {n_silver_valid/n_gold_eu:.0f} registros VALID/agregado anual")

print(f"\n✅ Pipeline Medallion completado — Lago Kasumigaura (1977–2024)")
print(f"   Bronze  → {n_bronze:,} registros raw")
print(f"   Silver  → {n_silver_valid:,} registros VALID ({n_silver_valid/n_silver*100:.1f}%)")
print(f"   Gold Ox → {n_gold_ox:,} índices mensuales de oxigenación")
print(f"   Gold Eu → {n_gold_eu:,} tendencias anuales de eutrofización")